In [0]:
import os

module_path = "/Workspace/Users/mhibar@aubay.com/data_quality.py"

print("File exists:", os.path.exists(module_path))

In [0]:
from pyspark.sql import functions as F

# Read the validated Silver Delta table
silver_df = spark.table("workspace.default.silver_wind_power")

# Verify the source data
print("Number of rows:", silver_df.count())
print("Number of columns:", len(silver_df.columns))

display(silver_df.limit(10))

In [0]:
# Vérifier les données Silver avant le calcul Gold
quality_report = run_quality_checks(silver_df)

number_of_rules = quality_report.count()

failed_rules = quality_report.filter(
    F.col("result") != "PASS"
).count()

print("Nombre de règles de qualité :", number_of_rules)
print("Nombre de règles en échec :", failed_rules)

# Bloquer le calcul Gold si les contrôles échouent
if number_of_rules != 8:
    raise ValueError(
        f"Traitement Gold bloqué : 8 règles attendues, "
        f"{number_of_rules} trouvées."
    )

if failed_rules > 0:
    raise ValueError(
        f"Traitement Gold bloqué : {failed_rules} règle(s) en échec."
    )

print("Contrôles qualité réussis. Calcul Gold autorisé.")

# Calcul des indicateurs Gold
gold_daily_df = (
    silver_df
    .groupBy("date", "turbine_name")
    .agg(
        F.sum("energy_produced").alias("total_energy_produced"),
        F.avg("wind_speed").alias("avg_wind_speed"),
        F.count("*").alias("total_measurements"),
        F.sum(
            F.when(F.col("status") == "Online", 1).otherwise(0)
        ).alias("online_measurements"),
        F.sum(
            F.when(F.col("status") == "Offline", 1).otherwise(0)
        ).alias("offline_measurements")
    )
    .orderBy("date", "turbine_name")
)

display(gold_daily_df.limit(10))

In [0]:
from pyspark.sql import functions as F

gold_daily_df = gold_daily_df.withColumn(
    "online_measurement_pct",
    F.round(
        F.col("online_measurements")
        / F.col("total_measurements") * 100,
        2
    )
)

display(gold_daily_df.limit(10))

In [0]:
# Validate the aggregated dataset
print("Number of rows:", gold_daily_df.count())
print("Number of columns:", len(gold_daily_df.columns))

display(gold_daily_df)

In [0]:
print("Number of rows:", gold_daily_df.count())
print("Number of columns:", len(gold_daily_df.columns))

display(gold_daily_df)

In [0]:
from pyspark.sql import functions as F

gold_table_name = "workspace.default.gold_daily_production"

# 1. Vérifier les contrôles qualité Silver
quality_report = run_quality_checks(silver_df)

number_of_rules = quality_report.count()

failed_rules = quality_report.filter(
    F.col("result") != "PASS"
).count()

if number_of_rules != 8:
    raise ValueError(
        f"Écriture Gold bloquée : 8 règles attendues, "
        f"{number_of_rules} trouvées."
    )

if failed_rules > 0:
    raise ValueError(
        f"Écriture Gold bloquée : {failed_rules} règle(s) en échec."
    )

# 2. Vérifier le schéma Gold
expected_columns = [
    "date",
    "turbine_name",
    "total_energy_produced",
    "avg_wind_speed",
    "total_measurements",
    "online_measurements",
    "offline_measurements",
    "online_measurement_pct",
]

if set(gold_daily_df.columns) != set(expected_columns):
    raise ValueError(
        "Écriture Gold bloquée : le schéma du DataFrame est incorrect."
    )

# 3. Vérifier la cohérence des indicateurs Gold
invalid_gold_rows = gold_daily_df.filter(
    (F.col("total_measurements") <= 0)
    | (F.col("online_measurements") > F.col("total_measurements"))
    | (F.col("offline_measurements") > F.col("total_measurements"))
    | (F.col("online_measurement_pct") < 0)
    | (F.col("online_measurement_pct") > 100)
).count()

if invalid_gold_rows > 0:
    raise ValueError(
        f"Écriture Gold bloquée : {invalid_gold_rows} "
        "groupe(s) invalide(s)."
    )

# 4. Écrire uniquement si tous les contrôles réussissent
print("Contrôles réussis. Écriture de la table Gold...")

gold_daily_df.write.format("delta").mode("overwrite").saveAsTable(
    gold_table_name
)

print("Table Gold mise à jour avec succès.")

In [0]:
# Read the saved Gold table

gold_saved_df = spark.table("workspace.default.gold_daily_production")

print("Number of rows:", gold_saved_df.count())
print("Number of columns:", len(gold_saved_df.columns))

display(gold_saved_df.limit(10))

In [0]:
# Save daily production metrics as a Delta table

gold_daily_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.gold_daily_production")

In [0]:
# Read the saved Gold table

gold_saved_df = spark.table("workspace.default.gold_daily_production")

print("Number of rows:", gold_saved_df.count())
print("Number of columns:", len(gold_saved_df.columns))

display(gold_saved_df.limit(10))

In [0]:
gold_daily_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.gold_daily_production")

In [0]:
gold_saved_df = spark.table("workspace.default.gold_daily_production")

In [0]:
# Check Gold table dimensions

print("Number of rows:", gold_saved_df.count())
print("Number of columns:", len(gold_saved_df.columns))

In [0]:
# Check uniqueness of date and turbine_name

duplicate_groups_df = (
    gold_saved_df
    .groupBy("date", "turbine_name")
    .count()
    .filter(F.col("count") > 1)
)

print("Number of duplicate groups:", duplicate_groups_df.count())

display(duplicate_groups_df)

In [0]:
# Check measurement count consistency

inconsistent_measurements_df = gold_saved_df.filter(
    F.col("online_measurements")
    + F.col("offline_measurements")
    > F.col("total_measurements")
)

print(
    "Number of inconsistent groups:",
    inconsistent_measurements_df.count()
)

display(inconsistent_measurements_df)

In [0]:
# Calculate the percentage of measurements recorded as Online

gold_daily_df = gold_daily_df.withColumn(
    "online_measurement_pct",
    F.round(
        F.col("online_measurements")
        / F.col("total_measurements") * 100,
        2
    )
)

display(gold_daily_df.limit(10))

In [0]:
print("Number of rows:", gold_daily_df.count())
print("Number of columns:", len(gold_daily_df.columns))

display(
    gold_daily_df.select(
        "date",
        "turbine_name",
        "total_measurements",
        "online_measurements",
        "online_measurement_pct"
    ).limit(10)
)

In [0]:
gold_daily_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.gold_daily_production")

In [0]:
%sql
ALTER TABLE workspace.default.gold_daily_production
ADD COLUMNS (online_measurement_pct DOUBLE)

In [0]:
gold_daily_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.gold_daily_production")

In [0]:
gold_saved_df = spark.table(
    "workspace.default.gold_daily_production"
)

print("Number of rows:", gold_saved_df.count())
print("Number of columns:", len(gold_saved_df.columns))

display(
    gold_saved_df.select(
        "date",
        "turbine_name",
        "total_measurements",
        "online_measurements",
        "online_measurement_pct"
    ).limit(10)
)

In [0]:
gold_turbine_df = (
    silver_df
    .groupBy("turbine_name")
    .agg(
        F.sum("energy_produced").alias("total_energy_produced"),
        F.avg("wind_speed").alias("avg_wind_speed"),
        F.count("*").alias("total_measurements"),
        F.sum(
            F.when(F.col("status") == "Online", 1).otherwise(0)
        ).alias("online_measurements"),
        F.sum(
            F.when(F.col("status") == "Offline", 1).otherwise(0)
        ).alias("offline_measurements")
    )
    .withColumn(
        "online_measurement_pct",
        F.round(
            F.col("online_measurements")
            / F.col("total_measurements") * 100,
            2
        )
    )
    .orderBy(F.col("total_energy_produced").desc())
)

display(gold_turbine_df)

In [0]:
gold_turbine_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.gold_turbine_performance")

In [0]:
gold_turbine_saved_df = spark.table(
    "workspace.default.gold_turbine_performance"
)

print("Nombre de lignes :", gold_turbine_saved_df.count())
print("Nombre de colonnes :", len(gold_turbine_saved_df.columns))

display(gold_turbine_saved_df)

In [0]:
duplicate_turbines_df = (
    gold_turbine_saved_df
    .groupBy("turbine_name")
    .count()
    .filter(F.col("count") > 1)
)

print(
    "Nombre de groupes de doublons :",
    duplicate_turbines_df.count()
)

display(duplicate_turbines_df)

In [0]:
duplicate_turbines_df = (
    gold_turbine_saved_df
    .groupBy("turbine_name")
    .count()
    .filter(F.col("count") > 1)
)

In [0]:
print(
    "Nombre de groupes de doublons :",
    duplicate_turbines_df.count()
)

In [0]:
display(duplicate_turbines_df)

In [0]:
# List tables in the default schema
display(spark.sql("SHOW TABLES IN workspace.default"))

In [0]:
# Inspect the schema of each table

tables = [
    "workspace.default.bronze_wind_power",
    "workspace.default.silver_wind_power",
    "workspace.default.gold_daily_production",
    "workspace.default.gold_turbine_performance"
]

for table_name in tables:
    print(f"\n{'=' * 60}")
    print(f"Table: {table_name}")
    print("=" * 60)

    spark.table(table_name).printSchema()

In [0]:
from pyspark.sql import functions as F
quality_history_df = spark.table(
    "workspace.default.data_quality_results"
)

display(
    quality_history_df
    .orderBy(F.col("execution_timestamp").desc_nulls_last())
)

In [0]:
from pyspark.sql import functions as F

quality_history_df = spark.table(
    "workspace.default.data_quality_results"
)

# Identify the latest quality check execution
latest_execution = (
    quality_history_df
    .filter(F.col("execution_timestamp").isNotNull())
    .agg(F.max("execution_timestamp").alias("latest_execution"))
    .first()["latest_execution"]
)

print("Latest quality check execution:", latest_execution)

# Keep only the results from that execution
latest_quality_results_df = quality_history_df.filter(
    F.col("execution_timestamp") == F.lit(latest_execution)
)

display(latest_quality_results_df)

In [0]:
# Count the quality checks and failures
number_of_rules = latest_quality_results_df.count()

failed_rules = latest_quality_results_df.filter(
    F.col("result") != "PASS"
).count()

print("Number of quality rules:", number_of_rules)
print("Failed quality rules:", failed_rules)

# Block Gold processing if quality checks fail
if number_of_rules != 8:
    raise ValueError(
        f"Expected 8 quality rules, but found {number_of_rules}."
    )

if failed_rules > 0:
    raise ValueError(
        f"Gold processing blocked: {failed_rules} quality rule(s) failed."
    )

print("Data quality validation successful. Gold processing can continue.")

In [0]:
import sys

module_dir = "/Workspace/Users/mhibar@aubay.com"

if module_dir not in sys.path:
    sys.path.insert(0, module_dir)

from data_quality import run_quality_checks

print("Function imported successfully:", callable(run_quality_checks))

In [0]:
import os
import sys

module_dir = "/Workspace/Users/mhibar@aubay.com"

print("Dossier accessible :", os.path.isdir(module_dir))
print("Contenu du dossier :", os.listdir(module_dir))
print("Dossier dans sys.path :", module_dir in sys.path)

In [0]:
module_path = "/Workspace/Users/mhibar@aubay.com/data_quality.py"

with open(module_path, "r") as file:
    print(file.read()[:500])

In [0]:
import os
import sys

print("Répertoire courant :", os.getcwd())
print("Chemin du module présent :", 
      "/Workspace/Users/mhibar@aubay.com" in sys.path)

print("Fichier visible :", os.path.isfile(
    "/Workspace/Users/mhibar@aubay.com/data_quality.py"
))

print("Début de sys.path :")
print("\n".join(sys.path[:10]))

In [0]:
import importlib.util

module_path = "/Workspace/Users/mhibar@aubay.com/data_quality.py"

# Load the Python module directly from its file path
spec = importlib.util.spec_from_file_location(
    "data_quality",
    module_path
)

if spec is None or spec.loader is None:
    raise ImportError(f"Unable to load module from {module_path}")

data_quality_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(data_quality_module)

# Retrieve the function
run_quality_checks = data_quality_module.run_quality_checks

print("Function imported successfully:", callable(run_quality_checks))

In [0]:
# Run the data quality checks on the Silver DataFrame
quality_report = run_quality_checks(silver_df)

# Display the results
display(quality_report)

In [0]:
# Count the executed rules
number_of_rules = quality_report.count()

# Count failed quality rules
failed_rules = quality_report.filter(
    F.col("result") != "PASS"
).count()

print("Number of quality rules:", number_of_rules)
print("Failed quality rules:", failed_rules)

# Validate the quality report before Gold processing
if number_of_rules != 8:
    raise ValueError(
        f"Expected 8 quality rules, but found {number_of_rules}."
    )

if failed_rules > 0:
    raise ValueError(
        f"Gold processing blocked: {failed_rules} quality rule(s) failed."
    )

print("Data quality validation successful. Gold processing can continue.")

In [0]:
from pyspark.sql import functions as F

# Create a fictitious record containing invalid energy
test_data = [
    (
        9001,
        "2024-06-01",
        "00:10:00",
        "Turbine A",
        150,
        "Wind Farm North",
        48.8566,
        2.3522,
        "North",
        "Online",
        "Operations",
        12.5,
        "N",
        -25.0,
        "2024-06-01 00:10:00",
    )
]

test_columns = [
    "production_id",
    "date",
    "time",
    "turbine_name",
    "capacity",
    "location_name",
    "latitude",
    "longitude",
    "region",
    "status",
    "responsible_department",
    "wind_speed",
    "wind_direction",
    "energy_produced",
    "measurement_timestamp",
]

test_df = spark.createDataFrame(
    test_data,
    schema=test_columns,
)

display(test_df)

In [0]:
# Run all quality checks on the fictitious data
test_quality_report = run_quality_checks(test_df)

display(test_quality_report)

In [0]:
test_quality_report = run_quality_checks(test_df)

display(test_quality_report)

In [0]:
test_failed_rules = test_quality_report.filter(
    F.col("result") != "PASS"
).count()

print("Nombre de règles en échec :", test_failed_rules)

if test_failed_rules > 0:
    print("TEST RÉUSSI : le traitement Gold doit être bloqué.")
else:
    raise AssertionError(
        "Le test aurait dû détecter au moins une règle en échec."
    )

In [0]:
# Vérifier la cohérence des indicateurs Gold
gold_quality_check = gold_daily_df.filter(
    (F.col("total_measurements") <= 0)
    | (F.col("online_measurements") > F.col("total_measurements"))
    | (F.col("offline_measurements") > F.col("total_measurements"))
    | (F.col("online_measurement_pct") < 0)
    | (F.col("online_measurement_pct") > 100)
)

print("Nombre de groupes Gold invalides :", gold_quality_check.count())

display(gold_quality_check)

In [0]:
gold_table_name = "workspace.default.gold_daily_production"

existing_gold_df = spark.table(gold_table_name)

print("Colonnes de la table Gold actuelle :")
existing_gold_df.printSchema()

print("Nombre de lignes :", existing_gold_df.count())
print("Nombre de colonnes :", len(existing_gold_df.columns))

In [0]:
# Comparer le schéma du DataFrame et celui de la table Gold
expected_columns = [
    "date",
    "turbine_name",
    "total_energy_produced",
    "avg_wind_speed",
    "total_measurements",
    "online_measurements",
    "offline_measurements",
    "online_measurement_pct",
]

dataframe_columns = gold_daily_df.columns
table_columns = existing_gold_df.columns

print("Colonnes manquantes dans le DataFrame :",
      sorted(set(expected_columns) - set(dataframe_columns)))

print("Colonnes inattendues dans le DataFrame :",
      sorted(set(dataframe_columns) - set(expected_columns)))

print("Schéma du DataFrame conforme :",
      set(dataframe_columns) == set(expected_columns))

print("Schéma de la table conforme :",
      set(table_columns) == set(expected_columns))

In [0]:
# Relire la table Gold depuis Delta
gold_saved_df = spark.table(
    "workspace.default.gold_daily_production"
)

# Vérifier le nombre de lignes et de colonnes
print("Nombre de lignes :", gold_saved_df.count())
print("Nombre de colonnes :", len(gold_saved_df.columns))

# Vérifier les colonnes attendues
print("Colonnes :", gold_saved_df.columns)

# Afficher quelques résultats
display(gold_saved_df.limit(10))

In [0]:
# Identifier les doublons dans Gold
gold_duplicates = (
    gold_saved_df
    .groupBy("date", "turbine_name")
    .count()
    .filter(F.col("count") > 1)
)

duplicate_groups = gold_duplicates.count()

print("Nombre de groupes en doublon :", duplicate_groups)

display(gold_duplicates)

In [0]:
# Comparer le DataFrame calculé avec la table Gold enregistrée
calculated_count = gold_daily_df.count()
saved_count = gold_saved_df.count()

print("Nombre de lignes calculées :", calculated_count)
print("Nombre de lignes enregistrées :", saved_count)

# Rechercher les lignes présentes dans le calcul,
# mais absentes de la table enregistrée
missing_in_saved = gold_daily_df.exceptAll(gold_saved_df)

# Rechercher les lignes enregistrées,
# mais absentes du calcul
extra_in_saved = gold_saved_df.exceptAll(gold_daily_df)

print("Lignes calculées absentes de la table :", missing_in_saved.count())
print("Lignes supplémentaires dans la table :", extra_in_saved.count())